<a href="https://colab.research.google.com/github/suryansh24-coder/Machine-Learning-Journey/blob/main/ML_Operations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=data.target_names)

print(f"Accuracy: {accuracy:.4f}")
print(report)

Accuracy: 1.0000
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=200, random_state=42),
    "Support Vector Machine": SVC(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    acc = accuracy_score(y_test, predictions)
    results[name] = acc

for name, score in results.items():
    print(f"{name}: {score:.4f}")

Logistic Regression: 1.0000
Support Vector Machine: 1.0000
Random Forest: 1.0000


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

extended_models = {
    "Logistic Regression": LogisticRegression(max_iter=200, random_state=42),
    "Support Vector Machine": SVC(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier()
}

for name, model in extended_models.items():
    scores = cross_val_score(model, X_train_pca, y_train, cv=5)
    model.fit(X_train_pca, y_train)
    test_acc = model.score(X_test_pca, y_test)
    print(f"{name}:")
    print(f"  Cross-Val Mean Accuracy: {np.mean(scores):.4f}")
    print(f"  Test Accuracy (with PCA): {test_acc:.4f}")

Logistic Regression:
  Cross-Val Mean Accuracy: 0.9167
  Test Accuracy (with PCA): 0.9000
Support Vector Machine:
  Cross-Val Mean Accuracy: 0.9250
  Test Accuracy (with PCA): 0.9000
Random Forest:
  Cross-Val Mean Accuracy: 0.9000
  Test Accuracy (with PCA): 0.9000
Gradient Boosting:
  Cross-Val Mean Accuracy: 0.9083
  Test Accuracy (with PCA): 0.8667
K-Nearest Neighbors:
  Cross-Val Mean Accuracy: 0.9250
  Test Accuracy (with PCA): 0.9333


In [4]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

param_grid_svm = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto'],
    'kernel': ['linear', 'rbf']
}

grid_search = GridSearchCV(SVC(random_state=42), param_grid_svm, cv=5)
grid_search.fit(X_train_pca, y_train)
best_svm = grid_search.best_estimator_

clf1 = LogisticRegression(max_iter=200, random_state=42)
clf2 = best_svm
clf3 = RandomForestClassifier(random_state=42)

ensemble = VotingClassifier(estimators=[('lr', clf1), ('svm', clf2), ('rf', clf3)], voting='hard')
ensemble.fit(X_train_pca, y_train)

ensemble_predictions = ensemble.predict(X_test_pca)
ensemble_accuracy = ensemble.score(X_test_pca, y_test)

conf_matrix = confusion_matrix(y_test, ensemble_predictions)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, ensemble_predictions, average='macro')

print(f"Best SVM Params: {grid_search.best_params_}")
print(f"Ensemble Accuracy: {ensemble_accuracy:.4f}")
print(f"Precision (Macro): {precision:.4f}")
print(f"Recall (Macro): {recall:.4f}")
print(f"F1 Score (Macro): {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)

Best SVM Params: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
Ensemble Accuracy: 0.9000
Precision (Macro): 0.9028
Recall (Macro): 0.8956
F1 Score (Macro): 0.8977
Confusion Matrix:
[[10  0  0]
 [ 0  7  2]
 [ 0  1 10]]
